### Investigating the files and preparing a sample submissin file detailing what should be mentioned and how to prepare such a file 


As with prior years, each game has a unique ID created by concatenating the season in which the game was played and the two team's respective TeamIds. For example, "2026_1101_1102" indicates a hypothetical matchup between team 1101 and 1102 in the year 2026. You must predict the probability that the team with the lower TeamId beats the team with the higher TeamId. Note that the men's teams and women's TeamIds do not overlap.

The resulting submission format looks like the following, where Pred represents the predicted probability that the first team will win:

ID,Pred
2026_1101_1102,0.5
2026_1101_1103,0.5
2026_1101_1104,0.5
...


In [2]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from src.data_loader import (
    load_teams, load_seasons, load_regular_season_results,
    load_tourney_results, load_seeds, validate_data_coverage
)
from src.utils import check_data_quality, get_season_summary

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## Regular season detailed processing

In [ ]:
men_regular = load_regular_season_results('M','detailed')
print(men_regular.columns)

## Column's for detailed processing

### Column Reference

**Game Info**

| Column | Description |
|--------|-------------|
| `Season` | Year of the season |
| `DayNum` | Days since DayZero (season start date) |
| `WTeamID` | Winning team's ID |
| `WScore` | Winning team's final score |
| `LTeamID` | Losing team's ID |
| `LScore` | Losing team's final score |
| `WLoc` | Winner's location — **H**ome, **A**way, **N**eutral |
| `NumOT` | Number of overtime periods (0 = regulation) |

**Shooting Stats**

| Stat | W (Winner) | L (Loser) | Description |
|------|------------|-----------|-------------|
| Field Goals Made | `WFGM` | `LFGM` | Total baskets made (2pt + 3pt) |
| Field Goals Attempted | `WFGA` | `LFGA` | Total shot attempts |
| 3-Pointers Made | `WFGM3` | `LFGM3` | 3-point baskets made |
| 3-Pointers Attempted | `WFGA3` | `LFGA3` | 3-point shot attempts |
| Free Throws Made | `WFTM` | `LFTM` | Free throws made |
| Free Throws Attempted | `WFTA` | `LFTA` | Free throw attempts |

**Rebounds**

| Stat | W (Winner) | L (Loser) | Description |
|------|------------|-----------|-------------|
| Offensive Rebounds | `WOR` | `LOR` | Grabbed after own team's missed shot |
| Defensive Rebounds | `WDR` | `LDR` | Grabbed after opponent's missed shot |

**Other Stats**

| Stat | W (Winner) | L (Loser) | Description |
|------|------------|-----------|-------------|
| Assists | `WAst` | `LAst` | Passes leading directly to a score |
| Turnovers | `WTO` | `LTO` | Lost possession to the opponent |
| Steals | `WStl` | `LStl` | Took the ball from opponent |
| Blocks | `WBlk` | `LBlk` | Deflected opponent's shot attempt |
| Personal Fouls | `WPF` | `LPF` | Fouls committed |

> **Prefix key:** `W` = Winner, `L` = Loser. Every stat is recorded for both teams.


In [ ]:
men_regular.head(10)

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR',
       'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3',
       'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'],
      dtype='str')


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,3,14,11,18,14,24,13,23,7,1,22,22,53,2,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,8,20,10,19,15,28,16,13,4,4,18,24,67,6,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,8,18,17,29,17,26,15,10,5,2,25,22,73,3,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,3,9,17,31,6,19,11,12,14,2,18,18,49,6,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,6,14,11,13,17,22,12,14,4,4,20,24,62,6,16,17,27,21,15,12,10,7,1,14
5,2003,11,1458,81,1186,55,H,0,26,57,6,12,23,27,12,24,12,9,9,3,18,20,46,3,11,12,17,6,22,8,19,4,3,25
6,2003,12,1161,80,1236,62,H,0,23,55,2,8,32,39,13,18,14,17,11,1,25,19,41,4,15,20,28,9,21,11,30,10,4,28
7,2003,12,1186,75,1457,61,N,0,28,62,4,14,15,21,13,35,19,19,7,2,21,20,59,4,17,17,23,8,25,10,15,14,8,18
8,2003,12,1194,71,1156,66,N,0,28,58,5,11,10,18,9,22,9,17,9,2,23,24,52,6,18,12,27,13,26,13,25,8,2,18
9,2003,12,1458,84,1296,56,H,0,32,67,5,17,15,19,14,22,11,6,12,0,13,23,52,3,14,7,12,9,23,10,18,1,3,18


In [ ]:
men_tourney = load_tourney_results('M','detailed')
print(men_tourney.columns)
men_tourney.head(10)

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR',
       'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3',
       'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'],
      dtype='str')


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,134,1421,92,1411,84,N,1,32,69,11,29,17,26,14,30,17,12,5,3,22,29,67,12,31,14,31,17,28,16,15,5,0,22
1,2003,136,1112,80,1436,51,N,0,31,66,7,23,11,14,11,36,22,16,10,7,8,20,64,4,16,7,7,8,26,12,17,10,3,15
2,2003,136,1113,84,1272,71,N,0,31,59,6,14,16,22,10,27,18,9,7,4,19,25,69,7,28,14,21,20,22,11,12,2,5,18
3,2003,136,1141,79,1166,73,N,0,29,53,3,7,18,25,11,20,15,18,13,1,19,27,60,7,17,12,17,14,17,20,21,6,6,21
4,2003,136,1143,76,1301,74,N,1,27,64,7,20,15,23,18,20,17,13,8,2,14,25,56,9,21,15,20,10,26,16,14,5,8,19
5,2003,136,1163,58,1140,53,N,0,17,52,4,14,20,27,12,29,8,14,3,8,16,20,64,2,17,11,13,15,26,11,11,8,4,22
6,2003,136,1181,67,1161,57,N,0,19,54,4,13,25,31,13,27,4,16,10,8,23,18,54,3,11,18,22,11,24,8,19,5,4,19
7,2003,136,1211,74,1153,69,N,0,20,47,6,14,28,37,8,28,12,12,2,2,15,26,66,10,27,7,10,13,22,13,10,7,6,24
8,2003,136,1228,65,1443,60,N,0,24,56,5,14,12,14,15,23,15,14,11,4,14,22,58,8,24,8,13,17,18,10,14,6,5,16
9,2003,136,1242,64,1429,61,N,0,28,51,2,6,6,11,7,20,13,11,8,4,17,23,56,6,17,9,10,13,19,13,13,6,1,15


In [4]:
women_regular = load_regular_season_results('W')
women_regular.head(10)

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1998,18,3104,91,3202,41,H,0
1,1998,18,3163,87,3221,76,H,0
2,1998,18,3222,66,3261,59,H,0
3,1998,18,3307,69,3365,62,H,0
4,1998,18,3349,115,3411,35,H,0
5,1998,18,3435,65,3172,63,H,0
6,1998,18,3443,89,3257,73,H,0
7,1998,19,3119,75,3217,47,H,0
8,1998,19,3139,74,3103,58,H,0
9,1998,19,3141,73,3387,55,H,0
